# Supplementary tables

Assembles the supplementary table sheets from the canonical datasets and the analysis
outputs. Each is written to `chapters/06-supplementary-tables/sheets/` as a CSV;
`build_workbook.py` collects them into one workbook.

| Table | Built here | Source |
| --- | --- | --- |
| ST2 discordant lead variants | yes | Results 4 |
| ST4 gPS against gene sets | yes | Results 5 |
| ST5 ChEMBL target-indication pairs | yes | Results 6 |
| ST6 drug target enrichment | yes | Results 6 |
| ST7 PAV gene-disease pairs with 2-5 areas | yes | Results 6 |
| ST9 therapeutic area assignment | yes | the hierarchy itself |
| ST14 gene-disease associations with gPS | yes | data preparation |
| ST15 cluster membership | yes | data preparation |
| ST16 disease distribution across areas | yes | data preparation |
| ST1 studies, ST10 fine-mapping, ST11 colocalisation | no | need the release study and colocalisation datasets; `02_tables_from_release.ipynb` |
| ST3 GSEA | no | static asset, see GAPS.md |
| ST12 L2G performance | no | `03_l2g_tables.ipynb` |
| ST18 effector genes, ST19 L2G training set | no | `05_effector_gene_list.ipynb`, `06_l2g_training_set.ipynb` |
| ST8 subgroup analysis | no | needs the therapeutic-area and target-class breakdown of Results 6 |

In [1]:
import pandas as pd
import pyarrow.dataset as ds

from manuscript_methods import clusters, paper

SHEETS = paper.ROOT / "chapters" / "06-supplementary-tables" / "sheets"
SHEETS.mkdir(parents=True, exist_ok=True)


def write(table: pd.DataFrame, name: str) -> None:
    """Write one sheet and report its shape."""
    path = SHEETS / f"{name}.csv"
    table.to_csv(path, index=False)
    print(f"{name}: {table.shape[0]} rows x {table.shape[1]} columns -> {path.name}")


names = clusters.disease_names()
areas = clusters.therapeutic_area_lookup()

# The counts the captions of tab:st2, tab:st5, tab:st14, tab:st15 and tab:st16 assert, registered
# in `tools/expected_numbers.tsv` as the `T*` rows and written out by the last cell.
numbers = {}

## ST1 — every GWAS study analysed

The study index restricted to `studyType == "gwas"`, with the columns the published sheet carries.
No derivation: this is the release table as ingested.

In [2]:
ST1_COLUMNS = [
    "studyId",
    "traitFromSource",
    "traitFromSourceMappedIds",
    "diseaseIds",
    "pubmedId",
    "publicationDate",
    "nCases",
    "nControls",
    "nSamples",
    "cohorts",
    "qualityControls",
    "analysisFlags",
]
st1 = ds.dataset(paper.release("study"), format="parquet").to_table(columns=ST1_COLUMNS + ["studyType"]).to_pandas()
st1 = st1[st1["studyType"] == "gwas"][ST1_COLUMNS].reset_index(drop=True)
write(st1, "ST1_studies")
print("published: 100526 rows x 12 columns")
st1.head(3)

ST1_studies: 100526 rows x 12 columns -> ST1_studies.csv
published: 100526 rows x 12 columns


,studyId,traitFromSource,traitFromSourceMappedIds,diseaseIds,pubmedId,publicationDate,nCases,nControls,nSamples,cohorts,qualityControls,analysisFlags
0,FINNGEN_R12_AUTOIMMUNE_HYPERTHYROIDISM,Autoimmune hyperthyroidism,[EFO_0004237],[EFO_0004237],None,None,2469.0,370637.0,373106.0,[FinnGen],[],[]
1,FINNGEN_R12_CD2_BENIGN_LARYNX,Benign neoplasm: Larynx,[MONDO_0002354],[MONDO_0002354],None,None,1152.0,499196.0,500348.0,[FinnGen],[],[]
2,FINNGEN_R12_E4_ADRENAS,"Disorders of adrenal gland, other and/or unspe...",[EFO_0005539],[EFO_0005539],None,None,167.0,479069.0,479236.0,[FinnGen],[],[]


## ST9 — therapeutic area assignment

In [3]:
st9 = pd.DataFrame(
    [{"EFO ID": root, "Therapeutic Area": label} for root, label in paper.THERAPEUTIC_AREAS.items()]
    + [{"EFO ID": "N/A", "Therapeutic Area": "other"}]
)
write(st9, "ST9_therapeutic_area_assignment")
st9

ST9_therapeutic_area_assignment: 24 rows x 2 columns -> ST9_therapeutic_area_assignment.csv


,EFO ID,Therapeutic Area
0,EFO_0001444,measurement
1,MONDO_0045024,cancer or benign tumor
2,OTAR_0000018,"genetic, familial or congenital disease"
3,EFO_0005741,infectious disease
4,OTAR_0000009,"injury, poisoning or other complication"
5,OTAR_0000014,pregnancy or perinatal disease
6,MONDO_0024458,disorder of visual system
7,EFO_0000319,cardiovascular disease
8,EFO_0009605,pancreas disease
9,EFO_0010282,gastrointestinal disease


## ST2 — lead variants with discordant pleiotropic effects

Variants with lead_vPS $\geq$ 10 whose directional concordance is $\leq$ 0.8, one row each.

**Universe: all 40,706 lead variants**, giving 37 rows over 29 colocalisation clusters and 37
distinct L2G-prioritised genes. `isClusterRepresentative` marks the 18 rows whose variant is its
cluster's representative — that subset is exactly the cluster-representative sheet this replaced, and
it is the set the main-text sentence is computed over (R4.17 67 representatives at lead_vPS $\geq$ 10,
R4.18 18 of them at concordance $\leq$ 0.8, R4.19 21 genes). Both universes are therefore readable
off the one sheet.

`cluster` is the colocalisation cluster id, carried so a reader can see where a locus contributes
more than one row: **cluster 121 contributes three TERT rows and its representative qualifies for
none of them**, and clusters 26, 103, 104, 307 and 350 contribute two each. Rows are ordered by
cluster, then concordance ascending, so those groups read together.

Selection is on the amended, sign-gated columns; `<= 0.8` matches what the main text now states, and
one row sits exactly at 0.8, so `< 0.8` would drop it.

**Which column each field reads.** Every count and score is amended-family; the ungated and published
columns are carried for provenance and used in no selection.

| sheet field | source |
| --- | --- |
| `geneSymbols`, `prioritisedGenes` | `prioritisedGenes` on `variant_features`, mapped to `approvedSymbol` through `gene_table` |
| `cluster`, `isClusterRepresentative` | `cluster_membership` for the variant-to-cluster map, `variant_clusters.leadVariantId` for the representative of each cluster |
| `diseases` | **`signedLeadVPS`** |
| `therapeuticAreas`, `therapeuticAreaNames` | **`signedLeadUniqueTherapeuticAreas`**, **`signedLeadTherapeuticAreas`** (Supplementary Table 9 hierarchy, the only one the pipeline carries) |
| `concordance` | **`signedLeadDirectionalConcordance`** |
| `diseasesIncreasedRisk`, `diseasesDecreasedRisk` | **`signedLeadUpDiseases`**, **`signedLeadDownDiseases`** |
| `exampleIncreasedRisk`, `exampleDecreasedRisk`, `allIncreasedRisk`, `allDecreasedRisk` | recomputed below from the **gated contributing credible sets** — single-disease studies with a non-null `directionOfEffect` — with the most significant study winning per disease |
| `leadVPS`, `leadDirectionalConcordance`, `uniqueDiseases`, `uniqueTherapeuticAreas`, `betaSignConcordance` | the first redefinition and the published columns, provenance only |

In [4]:
features = pd.read_parquet(paper.derived("variant_features"))

# Variant to cluster, and each cluster's representative. A lead variant belongs to exactly one
# cluster, so the map is a function.
membership = pd.read_parquet(paper.derived("cluster_membership"), columns=["cluster_id", "leadVariants"])
cluster_of = (
    membership.drop_duplicates("cluster_id")
    .assign(variantId=lambda d: d["leadVariants"].str.split(";"))
    .explode("variantId")
    .set_index("variantId")["cluster_id"]
)
assert not cluster_of.index.duplicated().any()
representative_of = pd.read_parquet(
    paper.derived("variant_clusters"), columns=["cluster_id", "leadVariantId"]
).set_index("cluster_id")["leadVariantId"]

# Every filter step, so the reduction to 37 rows is auditable.
step_defined = features[features["signedLeadVPSDefined"]]
step_high = step_defined[step_defined["signedLeadVPS"] >= 10]
discordant = step_high[step_high["signedLeadDirectionalConcordance"] <= 0.8].copy()
discordant["cluster"] = discordant["variantId"].map(cluster_of)
discordant["isClusterRepresentative"] = discordant["variantId"] == discordant["cluster"].map(representative_of)

print(
    pd.DataFrame(
        [
            {"step": "all lead variants", "rows": len(features)},
            {"step": "with a lead_vPS (something contributes)", "rows": len(step_defined)},
            {"step": "lead_vPS >= 10", "rows": len(step_high)},
            {"step": "concordance <= 0.8  (the sheet)", "rows": len(discordant)},
            {
                "step": "of those, the cluster representative  (R4.18)",
                "rows": int(discordant["isClusterRepresentative"].sum()),
            },
        ]
    )
    .set_index("step")
    .to_string()
)
assert discordant["cluster"].notna().all()
assert len(discordant) == 37, len(discordant)
assert discordant["cluster"].nunique() == 29, discordant["cluster"].nunique()
assert int(discordant["isClusterRepresentative"].sum()) == 18

# The flagged subset must be exactly the cluster-representative selection, which is what the
# main-text sentence is computed over.
representatives = set(
    pd.read_parquet(paper.derived("cluster_covariates"), columns=["clusterVariantId"])["clusterVariantId"]
)
assert set(discordant.loc[discordant["isClusterRepresentative"], "variantId"]) == set(
    discordant.loc[discordant["variantId"].isin(representatives), "variantId"]
)
assert len(representatives & set(step_high["variantId"])) == 67  # R4.17

# Duplicated loci, for the reader.
multi = discordant["cluster"].value_counts()
multi = multi[multi > 1]
print(f"\nclusters contributing more than one row: {len(multi)}")
for cluster_id, count in multi.sort_index().items():
    block = discordant[discordant["cluster"] == cluster_id]
    print(
        f"  cluster {cluster_id}: {count} rows | representative {representative_of.loc[cluster_id]}"
        f" | representative qualifies: {bool(block['isClusterRepresentative'].any())}"
    )

discordant = discordant.sort_values(
    ["cluster", "signedLeadDirectionalConcordance", "variantId"], ascending=[True, True, True]
)

                                                rows
step                                                
all lead variants                              40706
with a lead_vPS (something contributes)        35472
lead_vPS >= 10                                   118
concordance <= 0.8  (the sheet)                   37
of those, the cluster representative  (R4.18)     18

clusters contributing more than one row: 6
  cluster 26: 2 rows | representative 19_44888997_C_T | representative qualifies: False
  cluster 103: 2 rows | representative 9_133274295_A_T | representative qualifies: False
  cluster 104: 3 rows | representative 12_112379979_T_A | representative qualifies: True
  cluster 121: 3 rows | representative 5_1292868_C_A | representative qualifies: False
  cluster 307: 2 rows | representative 19_48702915_C_T | representative qualifies: True
  cluster 350: 2 rows | representative 14_94371805_G_T | representative qualifies: True


In [5]:
import numpy as np
import pyarrow.compute as pc

SELECTED = list(discordant["variantId"])

# The gated contributing credible sets of the selected variants: the study maps to exactly one
# disease term and the direction of effect is known. Same "most significant study wins per disease"
# rule and the same tie-break as `01-data-preparation/07_variant_features`.
associations = (
    ds.dataset(paper.derived("qualifying_credible_sets"), format="parquet")
    .to_table(
        columns={
            "variantId": ds.field("variantId"),
            "studyId": ds.field("studyId"),
            "studyLocusId": ds.field("studyLocusId"),
            "diseaseIds": ds.field("diseaseIds"),
            "direction": pc.struct_field(ds.field("rescaledStatistics"), "directionOfEffect"),
            "beta": pc.struct_field(ds.field("rescaledStatistics"), "absEstimatedBeta"),
            "pValueMantissa": pc.struct_field(ds.field("variantStatistics"), "pValueMantissa"),
            "pValueExponent": pc.struct_field(ds.field("variantStatistics"), "pValueExponent"),
        },
        filter=pc.field("variantId").isin(SELECTED),
    )
    .to_pandas()
)
gated = associations[
    associations["diseaseIds"].map(lambda ids: ids is not None and len(ids) == 1) & associations["direction"].notna()
].copy()
gated["diseaseId"] = gated["diseaseIds"].map(lambda ids: ids[0])
gated = gated.sort_values(["pValueExponent", "pValueMantissa", "studyLocusId"]).drop_duplicates(
    ["variantId", "diseaseId"]
)
gated["diseaseName"] = gated["diseaseId"].map(lambda d: names.get(d, d))
gated["negLog10P"] = -(np.log10(gated["pValueMantissa"].astype(float)) + gated["pValueExponent"].astype(float))
# Confirm the example associations can come from nowhere else: single-disease studies only, and
# only where the direction of effect is known.
assert gated["diseaseIds"].map(len).eq(1).all()
assert gated["direction"].notna().all()
print(f"gated contributing associations behind the {len(SELECTED)} rows: {len(gated):,}")

# Cross-check the recomputed directions against the columns the sheet reports, per variant.
recomputed = gated.groupby("variantId").agg(
    diseases=("diseaseId", "size"),
    up=("direction", lambda s: int((s > 0).sum())),
    down=("direction", lambda s: int((s < 0).sum())),
)
reported = (
    discordant.set_index("variantId")[["signedLeadVPS", "signedLeadUpDiseases", "signedLeadDownDiseases"]]
    .astype(int)
    .reindex(recomputed.index)
)
assert (recomputed["diseases"] == reported["signedLeadVPS"]).all()
assert (recomputed["up"] == reported["signedLeadUpDiseases"]).all()
assert (recomputed["down"] == reported["signedLeadDownDiseases"]).all()
print("recomputed disease and direction counts agree with the reported columns for all rows")

# A row that cannot illustrate discordance is one where the gate leaves nothing in a direction.
# It cannot happen at concordance <= 0.8 -- the minority share is at least 0.2 of a non-zero count --
# but it is checked rather than argued.
missing = recomputed[(recomputed["up"] == 0) | (recomputed["down"] == 0)]
print(f"rows with no gated association in one of the two directions: {len(missing)}")
assert missing.empty, missing


def examples(frame, sign, limit=2):
    """The most significant gated associations in one direction, as 'name (P = ...)'."""
    subset = frame[np.sign(frame["direction"]) == sign].nlargest(limit, "negLog10P")
    return "; ".join(
        f"{row.diseaseName} (P = {row.pValueMantissa:.1f}e{int(row.pValueExponent)})" for row in subset.itertuples()
    )


def all_names(frame, sign):
    """Every gated disease in one direction, most significant first."""
    subset = frame[np.sign(frame["direction"]) == sign].sort_values("negLog10P", ascending=False)
    return "; ".join(subset["diseaseName"])


per_variant = {
    variant: {
        "exampleIncreasedRisk": examples(block, 1),
        "exampleDecreasedRisk": examples(block, -1),
        "allIncreasedRisk": all_names(block, 1),
        "allDecreasedRisk": all_names(block, -1),
    }
    for variant, block in gated.groupby("variantId")
}

gated contributing associations behind the 37 rows: 617
recomputed disease and direction counts agree with the reported columns for all rows
rows with no gated association in one of the two directions: 0


In [6]:
symbols = pd.read_parquet(paper.derived("gene_table"), columns=["geneId", "approvedSymbol"])
symbol_of = dict(zip(symbols["geneId"], symbols["approvedSymbol"]))


def gene_symbols(genes):
    """L2G-prioritised genes as approved symbols, falling back to the id."""
    return "; ".join(sorted({symbol_of.get(g, g) for g in (genes if genes is not None else [])}))


def gene_ids(genes):
    """The same genes as Ensembl identifiers."""
    return "; ".join(sorted(genes if genes is not None else []))


# Columns and their order are the caption of tab:st2, with human-readable headers.
#
# "Number of associated diseases" is `signedLeadVPS`, the amended sign-gated lead_vPS the
# selection is made on. Two other disease counts sit on the frame and are dropped rather than
# shipped under headers a reader cannot tell apart: `leadVPS` (the first redefinition) and
# `uniqueDiseases` (the published column). `leadDirectionalConcordance`, `uniqueTherapeuticAreas`,
# `betaSignConcordance` and `therapeuticAreaNames` are dropped for the same reason -- none enters
# a selection here, and no manuscript number is read off this sheet.
#
# Three columns are kept past the caption list, each because a quoted count is read off it:
#   Cluster representative -- the 18 rows the main-text sentence is computed over
#   Cluster ID             -- what makes that flag checkable, and marks the six multi-row loci
#   Ensembl gene ID(s)     -- the 37 and 21 distinct-gene counts (R4.19) are over ids, not symbols
st2 = pd.DataFrame(
    {
        "Gene symbol(s)": discordant["prioritisedGenes"].map(gene_symbols),
        "Variant ID": discordant["variantId"],
        "Number of associated diseases": discordant["signedLeadVPS"].astype(int),
        "Number of associated therapeutic areas": discordant["signedLeadUniqueTherapeuticAreas"].astype(int),
        "Directional concordance": discordant["signedLeadDirectionalConcordance"].round(4),
        "Diseases with increased risk": discordant["signedLeadUpDiseases"].astype(int),
        "Diseases with decreased risk": discordant["signedLeadDownDiseases"].astype(int),
        "Example increased-risk associations": discordant["variantId"].map(
            lambda v: per_variant[v]["exampleIncreasedRisk"]
        ),
        "Example decreased-risk associations": discordant["variantId"].map(
            lambda v: per_variant[v]["exampleDecreasedRisk"]
        ),
        "All increased-risk associations": discordant["variantId"].map(lambda v: per_variant[v]["allIncreasedRisk"]),
        "All decreased-risk associations": discordant["variantId"].map(lambda v: per_variant[v]["allDecreasedRisk"]),
        "Cluster ID": discordant["cluster"].astype(int),
        "Cluster representative": discordant["isClusterRepresentative"],
        "Ensembl gene ID(s)": discordant["prioritisedGenes"].map(gene_ids),
    }
)
write(st2, "ST2_discordant_variants")

# The caption's three assertions, checked on the sheet itself rather than on the frame it came from.
assert len(st2) == 37, len(st2)
assert int(st2["Cluster representative"].sum()) == 18, int(st2["Cluster representative"].sum())
assert (st2["Number of associated diseases"] >= 10).all()
assert (st2["Directional concordance"] <= 0.8).all()

numbers["T2.01"] = len(st2)
numbers["T2.02"] = int(st2["Cluster representative"].sum())

genes_all = {g for gs in discordant["prioritisedGenes"] for g in (gs if gs is not None else [])}
flagged = discordant[discordant["isClusterRepresentative"]]
genes_reps = {g for gs in flagged["prioritisedGenes"] for g in (gs if gs is not None else [])}
assert len(genes_all) == 37, len(genes_all)
assert len(genes_reps) == 21, len(genes_reps)  # R4.19
print(
    pd.DataFrame(
        [
            {
                "universe": "all lead variants (the sheet)",
                "rows": len(st2),
                "clusters": int(st2["Cluster ID"].nunique()),
                "genes": len(genes_all),
            },
            {
                "universe": "cluster representatives (the main text)",
                "rows": len(flagged),
                "clusters": int(flagged["cluster"].nunique()),
                "genes": len(genes_reps),
            },
        ]
    )
    .set_index("universe")
    .to_string()
)
print("published sheet: 31 rows")
print(
    f"lead_vPS range: {st2['Number of associated diseases'].min()}-{st2['Number of associated diseases'].max()} | "
    f"concordance range: {st2['Directional concordance'].min():.4f}-{st2['Directional concordance'].max():.4f} | "
    f"rows at exactly 0.8: {int((st2['Directional concordance'] == 0.8).sum())}"
)
st2[
    [
        "Gene symbol(s)",
        "Variant ID",
        "Cluster ID",
        "Cluster representative",
        "Number of associated diseases",
        "Directional concordance",
        "Diseases with increased risk",
        "Diseases with decreased risk",
    ]
]

ST2_discordant_variants: 37 rows x 14 columns -> ST2_discordant_variants.csv
                                         rows  clusters  genes
universe                                                      
all lead variants (the sheet)              37        29     37
cluster representatives (the main text)    18        18     21
published sheet: 31 rows
lead_vPS range: 10-71 | concordance range: 0.5333-0.8000 | rows at exactly 0.8: 3


,Gene symbol(s),Variant ID,Cluster ID,Cluster representative,Number of associated diseases,Directional concordance,Diseases with increased risk,Diseases with decreased risk
26763,APOE,19_44906745_G_A,26,False,13,0.5385,7,6
11824,APOE,19_44908684_T_C,26,False,71,0.5634,40,31
17014,F5,1_169549811_C_T,30,False,21,0.7143,15,6
7735,PNPLA3,22_43928850_C_T,32,False,17,0.7647,13,4
26859,PHTF1; PTPN22,1_113761186_C_A,78,False,23,0.7391,17,6
35647,CCND2,12_4275678_T_G,90,True,17,0.6471,6,11
2991,ABCG8,2_43845437_G_T,100,True,13,0.6923,4,9
39671,ABO,9_133257521_T_TC,103,False,20,0.7000,14,6
29787,ABO,9_133274293_AC_A,103,False,12,0.7500,9,3
35559,ALDH2; TRAFD1,12_112136812_C_T,104,False,11,0.5455,5,6


## ST4 — gPS against membership in 21 gene sets

In [7]:
st4 = pd.read_csv(paper.derived("gene_pleiotropy_by_category.csv"))
write(st4, "ST4_gPS_gene_categories")
st4.head()

ST4_gPS_gene_categories: 21 rows x 10 columns -> ST4_gPS_gene_categories.csv


,category,label,odds_ratio,log_odds_ratio,ci_lower,ci_upper,log_ci_lower,log_ci_upper,p_value,fdr
0,Drosophila distant orthologs,Drosophila distant orthologs (830/41.9%),0.800112,-0.223004,0.732881,0.873510,-0.310773,-0.135236,6.361073e-07,1.113188e-06
1,Q1 LoF constraint,Q1 LoF constraint (4526/33.2%),0.854752,-0.156944,0.818083,0.893065,-0.200791,-0.113096,2.294544e-12,6.023177e-12
2,Essential Gene (DepMap),Essential Gene (DepMap) (1489/35.3%),0.860247,-0.150535,0.802621,0.922011,-0.219873,-0.081198,2.088882e-05,3.374348e-05
3,Non-essential Gene (DepMap),Non-essential Gene (DepMap) (766/31.5%),0.860816,-0.149874,0.778416,0.951939,-0.250494,-0.049254,3.507303e-03,5.260954e-03
4,Cellular lethal (FUSIL),Cellular lethal (FUSIL) (415/39.0%),0.875911,-0.132491,0.776079,0.988585,-0.253500,-0.011481,3.187947e-02,4.184180e-02


## ST5 — all ChEMBL target-indication pairs with genetic support

In [8]:
st5 = pd.read_csv(paper.derived("df_for_enrichment_regression.csv"))

# R2-MJ-17: the sheet shipped Ensembl and EFO identifiers only, so no reader could tell which
# target or indication a row is. Symbols come from the release target index and names from the
# release disease index -- the two sources ST7, ST14 and ST15 already join against -- and both
# cover every row, so the join adds no nulls and drops nothing.
target_symbols = (
    ds.dataset(paper.release("target"), format="parquet").to_table(columns=["id", "approvedSymbol"]).to_pandas()
)
st5["approvedSymbol"] = st5["targetId"].map(dict(zip(target_symbols["id"], target_symbols["approvedSymbol"])))
st5["diseaseName"] = st5["diseaseId"].map(names)
assert st5["approvedSymbol"].notna().all(), int(st5["approvedSymbol"].isna().sum())
assert st5["diseaseName"].notna().all(), int(st5["diseaseName"].isna().sum())

# Columns and their order are the caption of tab:st5. Every column the sheet carried is in the
# caption, so nothing is dropped.
ST5_COLUMNS = {
    "approvedSymbol": "Gene symbol",
    "targetId": "Ensembl target ID",
    "diseaseName": "Disease name",
    "diseaseId": "Disease ID",
    "indirect_assoc_score": "Open Targets indirect association score",
    "max_beta": "Maximum absolute rescaled effect size",
    "min_maf": "Minimum MAF",
    "max_vep": "Maximum VEP score",
    "maxClinicalPhase": "Highest clinical phase",
    "uniqueDiseases": "Number of unique associated diseases",
    "uniqueTherapeuticAreas": "Number of unique therapeutic areas",
    "outcome": "Approved (1 = approved)",
    "geneticSupport": "GWAS genetic support (1 = present)",
}
st5 = st5[list(ST5_COLUMNS)].rename(columns=ST5_COLUMNS)
write(st5, "ST5_chembl_ti_pairs")

# The caption's three counts.
approved = st5["Approved (1 = approved)"] == 1
supported = st5["GWAS genetic support (1 = present)"] == 1
print(
    f"rows: {len(st5):,} | approved: {int(approved.sum()):,} | approved with genetic support: {int((approved & supported).sum()):,}"
)
assert len(st5) == 37377, len(st5)
assert int(approved.sum()) == 4564, int(approved.sum())
assert int((approved & supported).sum()) == 242, int((approved & supported).sum())

numbers["T5.01"] = len(st5)
numbers["T5.02"] = int(approved.sum())
numbers["T5.03"] = int((approved & supported).sum())
st5.head()

ST5_chembl_ti_pairs: 37377 rows x 13 columns -> ST5_chembl_ti_pairs.csv
rows: 37,377 | approved: 4,564 | approved with genetic support: 242


,Gene symbol,Ensembl target ID,Disease name,Disease ID,Open Targets indirect association score,Maximum absolute rescaled effect size,Minimum MAF,Maximum VEP score,Highest clinical phase,Number of unique associated diseases,Number of unique therapeutic areas,Approved (1 = approved),GWAS genetic support (1 = present)
0,FGR,ENSG00000000938,HIV-1 infection,EFO_0000180,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0,0
1,FGR,ENSG00000000938,HIV infection,EFO_0000764,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0,0
2,FGR,ENSG00000000938,obesity,EFO_0001073,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0,0
3,FGR,ENSG00000000938,diabetic retinopathy,EFO_0003770,0.0,0.0,0.0,0.0,1.0,0.0,0.0,0,0
4,FGR,ENSG00000000938,chronic kidney disease,EFO_0003884,0.0,0.0,0.0,0.0,2.0,0.0,0.0,0,0


## ST6 — drug target enrichment results

In [9]:
forest = pd.read_csv(paper.derived("drug_enrichment_subsets_vs_full_l2g.csv"))
resources = pd.read_csv(paper.derived("drug_enrichment_other_resources.csv"))

# `total_indirect_assoc` is the size of the indirect-association universe an enrichment is computed
# in -- one constant per evidence source -- which is what `chemblDrugEnrichment` returns and what
# the `resources` rows carry (60,318 for OMIM, 15,304 for the gene-based tests). Results 6 builds
# the forest rows itself from the pair-level table, so it has no such column, and it renamed its
# per-stratum count of supported pairs into that slot: the published 151,704 read as 742 and one
# column held two different quantities across the sheet. The supported-pair count moves back to
# `n_support`, beside the `n_no_support` it pairs with, and the L2G universe is read off the
# temporal table's last year -- the same `chemblDrugEnrichment` call over the same full L2G
# evidence, so it is the same constant rather than a number typed in here.
temporal = pd.read_csv(paper.derived("temporal_drug_enrichment_full_chembl.csv"))
latest = temporal[temporal["datasource"] == temporal["datasource"].max()]
l2g_universe = int(latest["total_indirect_assoc"].unique().item())
forest = forest.rename(columns={"total_indirect_assoc": "n_support"})
forest.insert(forest.columns.get_loc("n_support"), "total_indirect_assoc", l2g_universe)
print(f"L2G indirect associations, from {latest['datasource'].iloc[0]}: {l2g_universe:,}")

st6 = pd.concat([forest, resources], ignore_index=True)
write(st6, "ST6_drug_target_enrichment")

# The published sheet's value, and the quantity that used to be reported under its name.
full = st6[(st6["datasource"] == "full_l2g") & (st6["clinicalPhase"] == "4+")]
print(
    f"full-dataset row: total_indirect_assoc {int(full['total_indirect_assoc'].iloc[0]):,} "
    f"| n_support {int(full['n_support'].iloc[0]):,}"
)
assert int(full["total_indirect_assoc"].iloc[0]) == 151704
assert int(full["n_support"].iloc[0]) == 742
assert st6["total_indirect_assoc"].notna().all()
st6[["datasource", "clinicalPhase", "odds_ratio", "Relative success", "yes_evid-high_clinphase"]].round(3)

L2G indirect associations, from 2025: 151,704
ST6_drug_target_enrichment: 35 rows x 21 columns -> ST6_drug_target_enrichment.csv
full-dataset row: total_indirect_assoc 151,704 | n_support 742


,datasource,clinicalPhase,odds_ratio,Relative success,yes_evid-high_clinphase
0,full_l2g,4+,3.619,2.765,242
1,PAV_subEvid,4+,6.048,3.791,72
2,PAV_base,4+,3.092,2.480,170
3,BigEffect_subEvid,4+,4.628,3.241,39
4,BigEffect_base,4+,3.473,2.689,203
5,rare_subEvid,4+,6.994,4.097,29
6,rare_base,4+,3.395,2.647,213
7,low-gPS-5_subEvid,4+,4.798,3.314,86
8,high-gPS_subEvid,4+,2.968,2.409,104
9,TA-1_subEvid,4+,4.291,3.064,22


## ST7 — gene-disease associations supported by a PAV with 2 to 5 therapeutic areas

One row per credible set supporting such an association, as published.

In [10]:
gene_table = pd.read_parquet(
    paper.derived("gene_table"), columns=["geneId", "approvedSymbol", "uniqueTherapeuticAreas"]
)
window = gene_table[gene_table["uniqueTherapeuticAreas"].between(2, 5)]

l2g = pd.read_parquet(
    paper.derived("prioritised_genes_diseases"),
    columns=[
        "geneId",
        "studyLocusId",
        "studyId",
        "score",
        "eQTL_coloc",
        "pQTL_coloc",
        "VEP",
        "distanceTSS",
        "variantId",
        "maf",
        "absBeta",
        "diseaseIds",
        "year",
    ],
)
st7 = l2g[(l2g["VEP"] == 1) & l2g["geneId"].isin(set(window["geneId"]))].merge(window, on="geneId", how="left")

# Ancestry and frequency flags, as published. Both derive exactly from the pipeline columns.
ancestry = pd.read_parquet(
    paper.derived("prioritised_genes_diseases"), columns=["geneId", "studyLocusId", "nfeFraction", "freqClass"]
)
st7 = st7.merge(ancestry, on=["geneId", "studyLocusId"], how="left")
st7["is_nfe"] = (st7["nfeFraction"] >= 0.9).astype(int)
st7["rare"] = (st7["freqClass"] == "rare").astype(int)
st7["nfe_common"] = ((st7["is_nfe"] == 1) & (st7["rare"] == 0)).astype(int)
st7["non_nfe_common"] = ((st7["is_nfe"] == 0) & (st7["rare"] == 0)).astype(int)
st7 = st7.drop(columns=["nfeFraction", "freqClass"])

# One row per credible set and disease, as published. `diseaseIds` is kept alongside as a
# semicolon-joined string: writing the numpy array straight to CSV produces "['A' 'B']", which
# reads back through ast.literal_eval as the single concatenated token 'AB'.
st7["diseaseIds"] = st7["diseaseIds"].apply(list)
st7 = st7.explode("diseaseIds").rename(columns={"diseaseIds": "diseaseId"})
st7.insert(
    st7.columns.get_loc("diseaseId"),
    "diseaseIds",
    st7.groupby(["geneId", "studyLocusId"])["diseaseId"].transform(lambda s: ";".join(sorted(set(s)))),
)
write(st7, "ST7_pav_gene_disease_pairs")

pairs = st7[["geneId", "diseaseId"]].dropna().drop_duplicates()
print("rows:", len(st7), "(published: 4742)")
print("distinct gene-disease associations:", len(pairs), "(manuscript: 2734)")
st7.head(3)

ST7_pav_gene_disease_pairs: 4742 rows x 20 columns -> ST7_pav_gene_disease_pairs.csv
rows: 4742 (published: 4742)
distinct gene-disease associations: 2734 (manuscript: 2734)


,geneId,studyLocusId,studyId,score,eQTL_coloc,pQTL_coloc,VEP,distanceTSS,variantId,maf,absBeta,diseaseIds,diseaseId,year,approvedSymbol,uniqueTherapeuticAreas,is_nfe,rare,nfe_common,non_nfe_common
0,ENSG00000184216,2db27a81275534ce51bb399852b3933c,FINNGEN_R12_AUTOIMMUNE,0.424932,1,0,1,1,X_154018741_A_G,0.152254,0.039383,EFO_0005140,EFO_0005140,2024,IRAK1,3,0,0,0,1
1,ENSG00000133466,0c79a22b919ee762992d4297c7afade7,FINNGEN_R12_AUTOIMMUNE,0.899293,1,0,1,1,22_37185445_C_A,0.390084,0.039157,EFO_0005140,EFO_0005140,2024,C1QTNF6,5,0,0,0,1
2,ENSG00000144802,d5924c489e8eb817d9dc364c7ac2ce6c,FINNGEN_R12_AUTOIMMUNE,0.906680,0,0,1,1,3_101852100_G_C,0.014385,0.114544,EFO_0005140,EFO_0005140,2024,NFKBIZ,4,0,0,0,1


## ST14 — every gene-disease association with gPS and area count

In [11]:
gene_table = pd.read_parquet(paper.derived("gene_table"))
associations = (
    pd.read_parquet(paper.derived("prioritised_genes_diseases"), columns=["geneId", "diseaseIds"])
    .explode("diseaseIds")
    .dropna()
    .drop_duplicates()
    .rename(columns={"diseaseIds": "diseaseId"})
)
st14 = associations.merge(
    gene_table[["geneId", "approvedSymbol", "uniqueDiseases", "uniqueTherapeuticAreas"]], on="geneId", how="left"
)
st14["diseaseName"] = st14["diseaseId"].map(names)
st14["therapeuticArea"] = st14["diseaseId"].map(lambda d: paper.THERAPEUTIC_AREAS.get(areas.get(d, "other"), "other"))

# Columns and their order are the caption of tab:st14. Every column the sheet carried is in the
# caption, so nothing is dropped.
ST14_COLUMNS = {
    "approvedSymbol": "Gene symbol",
    "geneId": "Ensembl gene ID",
    "diseaseName": "Disease name",
    "diseaseId": "EFO ID",
    "therapeuticArea": "Therapeutic area",
    "uniqueDiseases": "gPS",
    "uniqueTherapeuticAreas": "Number of therapeutic areas",
}
st14 = st14[list(ST14_COLUMNS)].rename(columns=ST14_COLUMNS)
write(st14, "ST14_gene_disease_with_gps")
print(
    f"rows: {len(st14):,} | unique genes: {st14['Ensembl gene ID'].nunique():,} | "
    f"unique diseases: {st14['EFO ID'].nunique():,}"
)
numbers["T14.01"] = len(st14)
numbers["T14.02"] = int(st14["Ensembl gene ID"].nunique())
numbers["T14.03"] = int(st14["EFO ID"].nunique())
st14.head()

ST14_gene_disease_with_gps: 36858 rows x 7 columns -> ST14_gene_disease_with_gps.csv
rows: 36,858 | unique genes: 8,285 | unique diseases: 1,394


,Gene symbol,Ensembl gene ID,Disease name,EFO ID,Therapeutic area,gPS,Number of therapeutic areas
0,TLR1,ENSG00000174125,Lyme disease,EFO_0008510,infectious disease,19,7
1,SCGB1D2,ENSG00000124935,Lyme disease,EFO_0008510,infectious disease,2,1
2,CRP,ENSG00000132693,bacterial disease,EFO_0000771,infectious disease,9,3
3,ADCY3,ENSG00000138031,bacterial disease,EFO_0000771,infectious disease,10,6
4,APOE,ENSG00000130203,bacterial disease,EFO_0000771,infectious disease,107,16


In [12]:
# Reconciliation against R1.31, the only published gene-disease association count (34,905, quoted
# in Results 1 and the Discussion as the denominator of the ancestry comparison). This sheet is the
# unrestricted list; R1.31 applies two restrictions the sheet deliberately does not.
pairs = (
    pd.read_parquet(paper.derived("prioritised_genes_diseases"), columns=["geneId", "diseaseIds", "year", "freqClass"])
    .explode("diseaseIds")
    .dropna(subset=["diseaseIds"])
    .rename(columns={"diseaseIds": "diseaseId"})
)


def distinct(frame):
    """Distinct gene-disease pairs."""
    return frame[["geneId", "diseaseId"]].drop_duplicates().shape[0]


MAX_YEAR = 2024  # `discovery.MAX_YEAR`: 2025 is a partial year in the release
common, recent = pairs["freqClass"] == "common", pairs["year"] <= MAX_YEAR
steps = pd.DataFrame(
    [
        {"restriction": "none -- this sheet", "pairs": distinct(pairs)},
        {"restriction": "common-variant evidence only", "pairs": distinct(pairs[common])},
        {"restriction": f"discovered by {MAX_YEAR}", "pairs": distinct(pairs[recent])},
        {"restriction": "both -- R1.31", "pairs": distinct(pairs[common & recent])},
    ]
).set_index("restriction")
print(steps.to_string())
assert distinct(pairs) == len(st14) == 36858
assert distinct(pairs[common & recent]) == 34905

only_rare = distinct(pairs) - distinct(pairs[common])
only_2025 = distinct(pairs) - distinct(pairs[recent])
dropped = distinct(pairs) - distinct(pairs[common & recent])
print(
    f"\n{dropped:,} pairs separate the two: {only_rare:,} have rare-variant evidence only, "
    f"{only_2025:,} were first seen in 2025 only, and {only_rare + only_2025 - dropped:,} fail both."
)

numbers["T14.04"] = only_rare
numbers["T14.05"] = only_2025

                              pairs
restriction                        
none -- this sheet            36858
common-variant evidence only  36213
discovered by 2024            35535
both -- R1.31                 34905

1,953 pairs separate the two: 645 have rare-variant evidence only, 1,323 were first seen in 2025 only, and 15 fail both.


## ST15 — diseases linked through each colocalisation cluster

In [13]:
st15 = pd.read_parquet(paper.derived("cluster_membership"))

# Columns and their order are the caption of tab:st15. Every column the sheet carried is in the
# caption, so nothing is dropped. Sorted on the cluster identifier -- R2-MJ-12 asks to inspect
# which diseases contribute to a given cluster, so each cluster has to read as one block; disease
# name orders within a cluster.
ST15_COLUMNS = {
    "cluster_id": "Cluster ID",
    "leadVariants": "Lead variant(s)",
    "diseaseName": "Disease name",
    "diseaseId": "EFO ID",
    "therapeuticArea": "Therapeutic area",
    "vPS": "vPS",
}
st15 = (
    st15[list(ST15_COLUMNS)]
    .rename(columns=ST15_COLUMNS)
    .sort_values(["Cluster ID", "Disease name", "EFO ID"])
    .reset_index(drop=True)
)
write(st15, "ST15_cluster_membership")

# The 2026-08-18 controls.
print(
    f"rows: {len(st15):,} | clusters: {st15['Cluster ID'].nunique():,} | "
    f"diseases: {st15['EFO ID'].nunique():,} | therapeutic areas: {st15['Therapeutic area'].nunique()} | "
    f"vPS range: {st15['vPS'].min()}-{st15['vPS'].max()}"
)
assert len(st15) == 42918, len(st15)
assert st15["Cluster ID"].nunique() == 20041
assert st15["EFO ID"].nunique() == 1403
assert st15["Therapeutic area"].nunique() == 23
assert (st15["vPS"].min(), st15["vPS"].max()) == (1, 120)

numbers["T15.01"] = len(st15)
numbers["T15.02"] = int(st15["Cluster ID"].nunique())
numbers["T15.03"] = int(st15["EFO ID"].nunique())
numbers["T15.04"] = int(st15["Therapeutic area"].nunique())
numbers["T15.05"] = int(st15["vPS"].max())
st15.head()

ST15_cluster_membership: 42918 rows x 6 columns -> ST15_cluster_membership.csv
rows: 42,918 | clusters: 20,041 | diseases: 1,403 | therapeutic areas: 23 | vPS range: 1-120


,Cluster ID,Lead variant(s),Disease name,EFO ID,Therapeutic area,vPS
0,0,16_89579029_G_T,melanoma,EFO_0000756,cancer or benign tumor,2
1,0,16_89579029_G_T,suntan,EFO_0004279,other,2
2,1,2_66523432_G_T,Sleep Disorder,EFO_0008568,nervous system disease,5
3,1,2_66523432_G_T,extrapyramidal and movement disease,MONDO_0001815,nervous system disease,5
4,1,2_66523432_G_T,insomnia,EFO_0004698,nervous system disease,5


## ST16 — distribution of diseases across therapeutic areas

Two universes: every disease term carried by a qualifying study, and the gPS disease list,
which is every disease term carrying at least one credible set with an L2G-prioritised gene.

In [14]:
qualifying_terms = set(
    ds.dataset(paper.derived("qualifying_gwas_studies"), format="parquet")
    .to_table(columns=["diseaseIds"])
    .to_pandas()["diseaseIds"]
    .explode()
    .dropna()
)
gps_terms = set(
    pd.read_parquet(paper.derived("prioritised_genes_diseases"), columns=["diseaseIds"])["diseaseIds"]
    .explode()
    .dropna()
)
print("qualifying disease terms:", len(qualifying_terms), "| gPS disease terms:", len(gps_terms))


def counts(terms):
    """Disease terms per therapeutic area, measurements excluded."""
    labels = pd.Series([areas.get(t, "other") for t in terms])
    return labels[labels != paper.MEASUREMENT].value_counts()


qualifying_counts, gps_counts = counts(qualifying_terms), counts(gps_terms)
st16 = pd.DataFrame(
    {
        "Root ID": list(paper.THERAPEUTIC_AREAS) + ["other"],
        "Therapeutic Area": list(paper.THERAPEUTIC_AREAS.values()) + ["other (no area root)"],
    }
)
st16 = st16[st16["Root ID"] != paper.MEASUREMENT]
st16["Diseases (qualifying dataset)"] = st16["Root ID"].map(qualifying_counts).fillna(0).astype(int)
st16["Diseases (gPS list)"] = st16["Root ID"].map(gps_counts).fillna(0).astype(int)
for column in ["qualifying dataset", "gPS list"]:
    total = st16[f"Diseases ({column})"].sum()
    st16[f"% of {column}"] = (100 * st16[f"Diseases ({column})"] / total).round(2)
st16 = st16.rename(
    columns={"% of qualifying dataset": "% of qualifying diseases", "% of gPS list": "% of gPS diseases"}
)
st16.insert(2, "Trait Class", "disease")

# Published layout: the measurement row first, then the therapeutic areas in hierarchy order, then
# a total over the disease side. Measurements have no gPS column because the gPS list is diseases.
measurement_row = {
    "Therapeutic Area": paper.THERAPEUTIC_AREAS[paper.MEASUREMENT],
    "Root ID": paper.MEASUREMENT,
    "Trait Class": "measurement",
    # Every trait carried by a qualifying measurement study, not only those whose ancestor is the
    # measurement root: the row is a trait class, and the Root ID on it is nominal.
    "Diseases (qualifying dataset)": len(
        set(
            ds.dataset(paper.derived("qualifying_measurement_studies"), format="parquet")
            .to_table(columns=["diseaseIds"])
            .to_pandas()["diseaseIds"]
            .explode()
            .dropna()
        )
    ),
}
total_row = {
    "Therapeutic Area": "TOTAL (disease side)",
    "Trait Class": "disease",
    "Diseases (qualifying dataset)": int(st16["Diseases (qualifying dataset)"].sum()),
    "Diseases (gPS list)": int(st16["Diseases (gPS list)"].sum()),
    "% of qualifying diseases": 100.00,
    "% of gPS diseases": 100.00,
}
st16 = pd.concat([pd.DataFrame([measurement_row]), st16, pd.DataFrame([total_row])], ignore_index=True)
st16 = st16[
    [
        "Therapeutic Area",
        "Root ID",
        "Trait Class",
        "Diseases (qualifying dataset)",
        "Diseases (gPS list)",
        "% of qualifying diseases",
        "% of gPS diseases",
    ]
]
for column in ["Diseases (qualifying dataset)", "Diseases (gPS list)"]:
    st16[column] = st16[column].astype("Int64")
write(st16, "ST16_ta_distribution")
st16

qualifying disease terms: 2320 | gPS disease terms: 1394
ST16_ta_distribution: 25 rows x 7 columns -> ST16_ta_distribution.csv


,Therapeutic Area,Root ID,Trait Class,Diseases (qualifying dataset),Diseases (gPS list),% of qualifying diseases,% of gPS diseases
0,measurement,EFO_0001444,measurement,7010,<NA>,NaN,NaN
1,cancer or benign tumor,MONDO_0045024,disease,350,240,15.09,17.22
2,"genetic, familial or congenital disease",OTAR_0000018,disease,180,121,7.76,8.68
3,infectious disease,EFO_0005741,disease,140,53,6.03,3.80
4,"injury, poisoning or other complication",OTAR_0000009,disease,64,38,2.76,2.73
5,pregnancy or perinatal disease,OTAR_0000014,disease,17,10,0.73,0.72
6,disorder of visual system,MONDO_0024458,disease,91,65,3.92,4.66
7,cardiovascular disease,EFO_0000319,disease,143,102,6.16,7.32
8,pancreas disease,EFO_0009605,disease,12,10,0.52,0.72
9,gastrointestinal disease,EFO_0010282,disease,89,53,3.84,3.80


### The hierarchy behind this sheet reproduces the gene-level assignment

The caption of `tab:st16` carried a `TODO(revisit)`: it said the therapeutic-area hierarchy this
sheet uses reproduces the study-level assignment but **not** the gene-level therapeutic-area
columns behind gPS and the therapeutic-area count, and asked whether the Supplementary Table 9
ordering does.

It does, and it is now the only ordering the pipeline carries. Aggregating the study-level
therapeutic-area membership to genes over `prioritised_genes_diseases` and comparing against the
pre-refactor `genes_therapeutic_areas` table, the Supplementary Table 9 ordering
(`primaryTherapeuticArea`) reproduces all 23 area columns and `totalStudies` for **0 of 8,285 genes
differing**. Under the legacy ordering, which `01-data-preparation/03_therapeutic_areas` used to
apply to the `paper.TA_COLUMNS` one-hot columns, 2,757 genes differed, `genetic, familial or
congenital disease` on 2,327 of them and `immune system disease` on 1,478. That ordering has been
deleted, so this check now has one arm.

`uniqueDiseases` (gPS) and `uniqueTherapeuticAreas` read `mappedTherapeuticAreas` and were already
on this ordering; they match the published gene-level table for 0 of 8,285 genes, which is what this
sheet, ST14, ST7's 2-5 therapeutic-area window and every gPS number rest on. See the chapter README.


In [15]:
# The gene-level therapeutic-area assignment under the single hierarchy, over all 8,285 genes.
baseline = pd.read_parquet(paper.baseline("genes_therapeutic_areas"))
gene_disease_studies = pd.read_parquet(paper.derived("prioritised_genes_diseases"), columns=["studyId", "geneId"])
study_areas = pd.read_parquet(paper.derived("study_therapeutic_areas"), columns=["studyId", "mappedTherapeuticAreas"])
AREA_COLUMNS = list(paper.TA_COLUMNS.values())

# Per-gene study counts per therapeutic area, one area per disease.
one_hot = pd.DataFrame(
    {
        column: study_areas["mappedTherapeuticAreas"].map(lambda a, r=root: int(a is not None and r in a))
        for root, column in paper.TA_COLUMNS.items()
    }
)
one_hot["studyId"] = study_areas["studyId"]
per_gene = gene_disease_studies.merge(one_hot, on="studyId", how="inner").groupby("geneId")[AREA_COLUMNS].sum()
per_gene = per_gene.assign(totalStudies=per_gene.sum(axis=1)).reset_index()

merged = per_gene.merge(
    baseline[["geneId", "totalStudies", *AREA_COLUMNS]], on="geneId", suffixes=("_new", "_baseline")
)
differing = pd.Series(False, index=merged.index)
per_column = {}
for column in ["totalStudies", *AREA_COLUMNS]:
    column_differs = merged[f"{column}_new"] != merged[f"{column}_baseline"]
    differing |= column_differs
    if column_differs.any():
        per_column[column] = int(column_differs.sum())
print(f"Supplementary Table 9 order: {int(differing.sum())} of {len(merged):,} genes differ")
for column, count in sorted(per_column.items(), key=lambda kv: -kv[1]):
    print(f"    {column}: {count}")
numbers["T16.01"] = int(differing.sum())
assert numbers["T16.01"] == 0
assert len(merged) == 8285, len(merged)

# gPS and the therapeutic-area count were already on this ordering.
gene_table = pd.read_parquet(
    paper.derived("gene_table"), columns=["geneId", "uniqueDiseases", "uniqueTherapeuticAreas"]
)
scores = gene_table.merge(
    baseline[["geneId", "uniqueDiseases", "uniqueTherapeuticAreas"]], on="geneId", suffixes=("_new", "_baseline")
)
for column in ["uniqueDiseases", "uniqueTherapeuticAreas"]:
    differing = int((scores[f"{column}_new"] != scores[f"{column}_baseline"]).sum())
    print(f"{column}: {differing} of {len(scores):,} genes differ")
    assert differing == 0

Supplementary Table 9 order: 0 of 8,285 genes differ
uniqueDiseases: 0 of 8,285 genes differ
uniqueTherapeuticAreas: 0 of 8,285 genes differ


## Numbers

The counts the captions assert.

In [16]:
print(paper.save_results("supplementary_tables", numbers))
pd.Series(numbers).to_frame("computed")

/Users/yt4/Projects/Gentropy-manuscript/results/supplementary_tables.json


,computed
T2.01,37
T2.02,18
T5.01,37377
T5.02,4564
T5.03,242
T14.01,36858
T14.02,8285
T14.03,1394
T14.04,645
T14.05,1323
